# Text-to-brain generation comparison

Compare MLP and CNN generation in their declared brain spaces on the same complete PubMed (3,066), Nilearn (79), and NeuroVault (202) test splits. CNN mixed-baseline heads are selected unless fine-tuned variants are requested explicitly.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
from neurovlm import AtlasFreeCNNDataProvider
from neurovlm.evaluation import (
    default_comparison_matrix, evaluate_text_to_brain_comparison,
)

DOMAINS = ("pubmed", "nilearn", "neurovault")
LIMIT_PER_DOMAIN = None  # full test split; set an integer only for a quick run
INCLUDE_FINETUNED = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EVALUATION_SCOPE = (
    "full test split"
    if LIMIT_PER_DOMAIN is None
    else f"first {LIMIT_PER_DOMAIN} paired examples per domain"
)


In [ ]:
results = []
for domain in DOMAINS:
    selections = default_comparison_matrix(
        "text_to_brain", domains=(domain,), include_finetuned=INCLUDE_FINETUNED
    )
    results.append(evaluate_text_to_brain_comparison(
        selections=selections,
        provider=AtlasFreeCNNDataProvider(domain=domain, limit=LIMIT_PER_DOMAIN),
        device=DEVICE,
    ))
summary = pd.DataFrame(row for result in results for row in result.summary)
by_source = pd.DataFrame(row for result in results for row in result.by_source)
by_sample = pd.DataFrame(row for result in results for row in result.by_sample)
manifest = pd.DataFrame(row for result in results for row in result.manifest)
summary.sort_values(["evaluation_domain", "family", "variant"])

`summary` contains spatial reconstruction metrics; `by_source` exposes the same metrics per corpus. This is a paired atlas-free comparison, but MLP and CNN outputs remain in different declared brain spaces. Compare trends and within-family domain changes; do not interpret their raw MSE values as voxel-identical measurements.

## Aggregate metrics

Lower is better for reconstruction MSE; higher is better for spatial correlation and top-5% Dice overlap.

In [ ]:
resolved = summary[(summary["status"] == "resolved") & (summary["n"] > 0)].copy()
if resolved.empty:
    raise RuntimeError("No models resolved. Inspect `manifest` for checkpoint errors.")
resolved["model"] = resolved.apply(
    lambda row: f'{row["family"].upper()} · {str(row["variant"]).replace("_", " ")}',
    axis=1,
)

metrics = (
    ("reconstruction_mse", "Reconstruction MSE ↓"),
    ("spatial_corr", "Spatial correlation ↑"),
    ("top5_dice", "Top-5% Dice ↑"),
)
fig, axes = plt.subplots(1, len(metrics), figsize=(18, 4.8))
for ax, (metric, title) in zip(axes, metrics):
    table = resolved.pivot(
        index="evaluation_domain", columns="model", values=metric
    ).reindex(DOMAINS)
    table.plot.bar(ax=ax, rot=0)
    ax.set_title(title)
    ax.set_xlabel("Evaluation domain")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(title="Model", fontsize=8)
fig.suptitle(f"Text-to-brain generation ({EVALUATION_SCOPE})")
fig.tight_layout()
plt.show()

## Per-sample overlap distributions

The distributions expose variation hidden by the aggregate means.

In [ ]:
sample_rows = by_sample[by_sample["status"] == "resolved"].copy()
sample_rows["model"] = sample_rows.apply(
    lambda row: f'{row["family"].upper()} · {str(row["variant"]).replace("_", " ")}',
    axis=1,
)
models = list(dict.fromkeys(sample_rows["model"]))
fig, axes = plt.subplots(1, len(DOMAINS), figsize=(18, 4.8), sharey=True)
for ax, domain in zip(axes, DOMAINS):
    domain_rows = sample_rows[sample_rows["evaluation_domain"] == domain]
    values = [domain_rows.loc[domain_rows["model"] == model, "top5_dice"] for model in models]
    ax.boxplot(values, tick_labels=models, showmeans=True)
    ax.set_title(domain.title())
    ax.tick_params(axis="x", rotation=20)
    ax.grid(axis="y", alpha=0.25)
axes[0].set_ylabel("Per-sample top-5% Dice")
fig.suptitle("Text-to-brain overlap distributions")
fig.tight_layout()
plt.show()

Use `ComparisonSelection(from_run=...)` when evaluating a local run rather than a released Hugging Face checkpoint. Mixed-baseline CNN heads remain the default; fine-tuned heads are included only when `INCLUDE_FINETUNED` is set explicitly.